In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

In [ ]:
# =============================================================================
def chuyen_anh_xam(img):
    """
    Chuyển ảnh màu (BGR) sang ảnh xám.
    Công thức chuyển đổi (chuẩn ITU-R BT.601):
        Gray = 0.299 * R + 0.587 * G + 0.114 * B
    """
    # Nếu ảnh đã là ảnh xám (chỉ có 2 chiều), trả về luôn
    if len(img.shape) == 2:
        return img.copy()
    
    # Nếu ảnh có 3 kênh màu, dùng OpenCV để chuyển BGR -> Gray
    # Bên trong, OpenCV dùng công thức: Gray = 0.299*R + 0.587*G + 0.114*B
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray


In [ ]:
url_img_test_cell2 = "input_images/image1.jpg"
img_mau_cell2 = cv2.imread(url_img_test_cell2)
anh_xam_cell2 = chuyen_anh_xam(img_mau_cell2)
#print(img_mau_cell2)
print("Kích thước ảnh màu gốc:", img_mau_cell2.shape)
print("Kích thước ảnh sau khi chuyển xám:", anh_xam_cell2.shape)

[[[  9  10   8]
  [  1   1   1]
  [ 17  16  18]
  ...
  [ 29 127  91]
  [ 52 156 109]
  [ 35 141  88]]

 [[  4   5   3]
  [  0   0   0]
  [  0   0   1]
  ...
  [ 36 135  97]
  [ 41 144  99]
  [ 40 146  99]]

 [[  3   4   0]
  [ 12  13  11]
  [  5   4   8]
  ...
  [ 45 141 100]
  [ 42 143 105]
  [ 39 144 107]]

 ...

 [[  7   8  12]
  [ 10  12  13]
  [  0   1   0]
  ...
  [ 48 146 110]
  [ 46 145 107]
  [ 35 135  93]]

 [[  8   9  13]
  [ 16  15  17]
  [ 10   8   7]
  ...
  [ 38 138 102]
  [ 48 144 107]
  [ 48 143 106]]

 [[  2   1   3]
  [  0   0   0]
  [ 16  14  13]
  ...
  [ 50 152 117]
  [ 31 121  91]
  [ 48 134 104]]]
Kích thước ảnh màu gốc: (151, 192, 3)
Kích thước ảnh sau khi chuyển xám: (151, 192)


In [ ]:
def tinh_histogram(gray_img):
    # Tạo mảng histogram với 256 phần tử, ban đầu bằng 0
    hist = np.zeros(256, dtype=np.int64)
    
    # Duyệt qua tất cả pixel trong ảnh
    # flatten() chuyển ảnh 2D thành mảng 1D để dễ duyệt
    for gia_tri_pixel in gray_img.flatten():
        hist[gia_tri_pixel] += 1
    
    return hist




In [ ]:
def ve_histogram(gray_img, tieu_de, duong_dan_luu):
    # Tính histogram
    hist = tinh_histogram(gray_img)
    
    # Tạo figure với kích thước 10x4 inch
    plt.figure(figsize=(10, 4))
    
    # Vẽ biểu đồ cột
    plt.bar(range(256), hist, color='steelblue', width=1.0)
    
    # Đặt tiêu đề và nhãn trục
    plt.title(tieu_de, fontsize=14)
    plt.xlabel('Muc xam (0-255)', fontsize=11)
    plt.ylabel('So luong pixel', fontsize=11)
    plt.xlim([0, 255])
    
    # Lưu biểu đồ
    plt.tight_layout()
    plt.savefig(duong_dan_luu, dpi=150, bbox_inches='tight')
    plt.close()  # Đóng figure để giải phóng bộ nhớ




In [ ]:
# =============================================================================
def can_bang_histogram(gray_img):
    # Lấy kích thước ảnh
    chieu_cao, chieu_rong = gray_img.shape
    tong_pixel = chieu_cao * chieu_rong
    
    # Bước 1: Tính histogram
    hist = tinh_histogram(gray_img)
    
    # Bước 2: Tính CDF (hàm phân phối tích lũy)
    # CDF[i] = tổng số pixel có mức xám từ 0 đến i
    cdf = np.cumsum(hist)
    
    # Bước 3: Chuẩn hóa CDF về khoảng [0, 255]
    # Công thức: muc_xam_moi = round(CDF[i] * 255 / tổng_pixel)
    cdf_chuan_hoa = np.round((cdf * 255.0) / tong_pixel).astype(np.uint8)
    
    # Bước 4: Ánh xạ từng pixel sang mức xám mới
    # Dùng kỹ thuật look-up table: pixel mới = bảng_ánh_xạ[pixel cũ]
    anh_can_bang = cdf_chuan_hoa[gray_img]
    
    return anh_can_bang




In [ ]:
# =============================================================================
def thu_hep_muc_xam(gray_img, muc_xam_min_moi=30, muc_xam_max_moi=120):
    
    # Tìm giá trị min và max hiện tại của ảnh
    gia_tri_min_cu = float(gray_img.min())
    gia_tri_max_cu = float(gray_img.max())
    
    # Trường hợp đặc biệt: nếu ảnh chỉ có 1 mức xám duy nhất
    if gia_tri_max_cu == gia_tri_min_cu:
        return np.full_like(gray_img, (muc_xam_min_moi + muc_xam_max_moi) // 2)
    
    # Áp dụng công thức ánh xạ tuyến tính
    # pixel_moi = (pixel_cu - min_cu) * (max_moi - min_moi) / (max_cu - min_cu) + min_moi
    ket_qua = (muc_xam_max_moi - muc_xam_min_moi) / (gia_tri_max_cu - gia_tri_min_cu) * (gray_img.astype(np.float64) - gia_tri_min_cu) + muc_xam_min_moi
    return ket_qua.astype(np.uint8)




In [ ]:
# =============================================================================
def xu_ly_histogram(img, ten_anh, thu_muc_output):
    # Tạo thư mục output nếu chưa có
    os.makedirs(thu_muc_output, exist_ok=True)
    
    # ---- Bước 1: Chuyển ảnh sang xám ----
    anh_xam = chuyen_anh_xam(img)
    duong_dan_anh_xam = os.path.join(thu_muc_output, f"{ten_anh}_gray.png")
    cv2.imwrite(duong_dan_anh_xam, anh_xam)
    print(f"  [Histogram] Da luu anh xam: {duong_dan_anh_xam}")
    
    # ---- Bước 2: Vẽ histogram H1 (ảnh xám gốc) ----
    duong_dan_h1 = os.path.join(thu_muc_output, f"{ten_anh}_H1_histogram.png")
    ve_histogram(anh_xam, f"H1 - Histogram anh xam goc ({ten_anh})", duong_dan_h1)
    print(f"  [Histogram] Da luu H1: {duong_dan_h1}")
    
    # ---- Bước 3: Cân bằng histogram ----
    anh_can_bang = can_bang_histogram(anh_xam)
    duong_dan_can_bang = os.path.join(thu_muc_output, f"{ten_anh}_equalized.png")
    cv2.imwrite(duong_dan_can_bang, anh_can_bang)
    print(f"  [Histogram] Da luu anh can bang: {duong_dan_can_bang}")
    
    # ---- Bước 4: Vẽ histogram H2 (sau cân bằng) ----
    duong_dan_h2 = os.path.join(thu_muc_output, f"{ten_anh}_H2_histogram.png")
    ve_histogram(anh_can_bang, f"H2 - Histogram sau can bang ({ten_anh})", duong_dan_h2)
    print(f"  [Histogram] Da luu H2: {duong_dan_h2}")
    
    # ---- Bước 5: Thu hẹp mức xám về [30, 120] ----
    anh_thu_hep = thu_hep_muc_xam(anh_can_bang, 30, 120)
    duong_dan_thu_hep = os.path.join(thu_muc_output, f"{ten_anh}_narrowed_30_120.png")
    cv2.imwrite(duong_dan_thu_hep, anh_thu_hep)
    print(f"  [Histogram] Da luu anh thu hep: {duong_dan_thu_hep}")
    
    # ---- Bước 6: Vẽ histogram sau thu hẹp ----
    duong_dan_h3 = os.path.join(thu_muc_output, f"{ten_anh}_H3_narrowed_histogram.png")
    ve_histogram(anh_thu_hep, f"Histogram sau thu hep [30,120] ({ten_anh})", duong_dan_h3)
    print(f"  [Histogram] Da luu histogram thu hep: {duong_dan_h3}")
    